In [1]:
import pandas as pd

In [2]:
# Load Dataset
df = pd.read_csv("investment_portfolio_dirty.csv")

In [3]:
# Remove duplicate rows
df = df.drop_duplicates()

In [4]:
df["market_cap_usd"] = pd.to_numeric(
    df["market_cap_usd"],
    errors="coerce"
)

In [7]:
# Convert market_cap_usd to float
df["market_cap_usd"] = pd.to_numeric(
    df["market_cap_usd"], errors="coerce"
).astype(float)

print(df["market_cap_usd"].head())
print(df["market_cap_usd"].dtype)

0    6.215190e+08
1             NaN
2    4.340732e+08
3    1.637663e+09
4    4.639344e+09
Name: market_cap_usd, dtype: float64
float64


In [8]:
# Remove leading/trailing spaces
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

In [9]:
# Standardize text format (First Letter Capital)
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].str.title()

In [10]:
# Standardize analyst ratings (Only Buy & Sell)
rating_map = {
    "Strong Buy": "Buy",
    "Buy": "Buy",
    "Hold": "Sell",
    "Sell": "Sell",
    "Underperform": "Sell"
}
df["analyst_rating"] = (
    df["analyst_rating"]
    .str.strip()
    .str.title()
    .replace(rating_map)
)

In [11]:
# Standardize ticker symbols
df["ticker"] = df["ticker"].str.upper()

In [12]:
# Standardize sector names
sector_map = {
    "Tech": "Technology",
    "Technology": "Technology",
    "Fin": "Financial",
    "Finance": "Financial"
}
df["sector"] = df["sector"].replace(sector_map)

In [14]:
# Convert date column
df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce")

In [15]:
currency_cols = ["open", "high", "low", "close", "market_cap_usd"]

for col in currency_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[$€,]", "", regex=True)
            .str.replace("usd", "", case=False, regex=True)
            .str.replace("M", "e6", regex=False)
            .str.replace("B", "e9", regex=False)
        )

        df[col] = df[col].apply(
            lambda x: eval(x) if isinstance(x, str) and ("e6" in x or "e9" in x) else x
        )

        df[col] = pd.to_numeric(df[col], errors="coerce")

In [16]:
# Convert numeric columns
numeric_cols = ["volume", "shares_held"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [17]:
# Fix OHLC errors
df.loc[df["low_price"] > df["high_price"], ["low_price", "high_price"]] = (
    df.loc[df["low_price"] > df["high_price"], ["high_price", "low_price"]].values
)
df.loc[df["high_price"] < df["close_price"], "high_price"] = df["close_price"]

In [18]:
# Fill missing values
df = df.fillna(df.median(numeric_only=True))
df = df.fillna(df.mode().iloc[0])

In [19]:
# Drop completely blank columns
df = df.dropna(axis=1, how="all")

In [21]:
# Save cleaned dataset
df.to_csv("Cleaned_investment_portfolio.csv", index=False)